In [3]:
import numpy as np
import matplotlib.pyplot as plt

from lr_utils import load_dataset

In [3]:
train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes = load_dataset()

In [220]:
m = train_set_x_orig.shape[0]
nx = train_set_x_orig.shape[1] * train_set_x_orig.shape[2] * train_set_x_orig.shape[3]
print("m =", m)
print("nx =", nx)
print("train_set_y_orig.shape", train_set_y_orig.shape)
print("train_set_x_orig.shape", train_set_y_orig.shape)

print(train_set_y_orig)

m = 209
nx = 12288
train_set_y_orig.shape (1, 209)
train_set_x_orig.shape (1, 209)
[[0 0 1 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 1 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0
  0 0 1 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 1 0 1 1 0 1 1 1 0 0 0 0 0 0 1 0 0 1
  0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 1 1 0 0 1 0 0 0 0 1 0 1 0 1 1
  1 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 1 0 1 0 1 1 0 0 0 1 1 1 1 1 0 0 0 0 1 0
  1 1 1 0 1 1 0 0 0 1 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0 1 0 1
  0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0]]


In [4]:
def initialize(nx):
    w = np.zeros((nx, 1))
    b = np.zeros((1, 1))
    return { "w": w, "b": b }

In [5]:
m_test = 209
nx_test = 12288

params_test = initialize(nx_test)

assert params_test["w"].shape == (nx_test, 1)
assert params_test["b"].shape == (1, 1)

In [6]:
def linear(X, w, b):
    return np.dot(w.T, X) + b

In [7]:
x_test = np.array([[1],
                   [2],
                   [3],
                   [4]])
w_test = np.array([[1],
                   [3],
                   [1],
                   [2]])
b_test = np.array([[1]])

z_test = linear(x_test, w_test, b_test)

assert np.array_equal(z_test, np.array([[19]]))

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [9]:
z_test = np.array([[19]])

a_test = sigmoid(z_test)

assert np.allclose(a_test, np.array([[0.99999999]]))

In [10]:
def cross_entropy(a, y):
    m = y.shape[1]
    return -(1/m) * np.sum(y * np.log(a) + (1 - y) * np.log(1 - a))

In [17]:
a_test3 = np.array([[0.75, 0.77]])
y_test3 = np.array([[1, 1]])
L3 = cross_entropy(a_test3, y_test3)
print(L3)

assert np.allclose(L3, np.array([[0.27452341829309423]]))

0.27452341829309423


In [18]:
def backprop(x, y, a):
    m = x.shape[1]
    dw = (1 / m) * np.dot(x, (a - y).T)
    db = (1 / m) * np.sum(a - y)

    return {
        'dw': dw,
        'db': db,
    }

In [20]:
x_test = np.array([[2]])
y_test = np.array([[1]])
a_test = np.array([[0.731]])

grads_test = backprop(x_test, y_test, a_test)

assert grads_test['dw'] == -0.538
assert grads_test['db'] == -0.269

In [21]:
def update_params(lr, w, b, dw, db):
    w = w - lr * dw
    b = b - lr * db
    return {
        'w': w,
        'b': b,
    }

In [22]:
lr_test = 0.1
w_test = np.array([1, 2, 3])
b_test = np.array([1, 1, 1])
dw_test = np.array([1, 2, 3])
db_test = np.array([1, 1, 1])

params_test = update_params(lr_test, w_test, b_test, dw_test, db_test)

assert np.array_equal(params_test['w'], np.array([0.9, 1.8, 2.7]))
assert np.array_equal(params_test['b'], np.array([0.9, 0.9, 0.9]))

In [23]:
def optimize(X, Y, lr, epoch):
    nx = X.shape[0]
    m = Y.shape[1]
    params = initialize(nx)
    losses = []

    for n in range(epoch):
        z = linear(X, params["w"], params["b"])
        a = sigmoid(z)
        loss = cross_entropy(a, Y)
        print("loss", loss)
        grads = backprop(X, Y, a)
        params = update_params(lr, params["w"], params["b"], grads['dw'], grads['db'])
        losses.append(loss)

    return params

In [25]:
def predict(w, b, X):
    m = X.shape[1]
    Y_prediction = np.zeros((1, m))
    w = w.reshape(X.shape[0], 1)

    A = sigmoid(np.dot(w.T, X) + b)

    for i in range(A.shape[1]):

        if A[0, i] > 0.5 :
            Y_prediction[0,i] = 1
        else:
            Y_prediction[0,i] = 0

    return Y_prediction

In [26]:
def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5):
    params = optimize(X_train, Y_train, learning_rate, num_iterations)

    w = params["w"]
    b = params["b"]

    Y_prediction_test = predict(w, b, X_test)
    Y_prediction_train = predict(w, b, X_train)

    print("train accuracy: {} %".format(100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100))
    print("test accuracy: {} %".format(100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100))

    return

In [27]:
train_set_x_orig, train_set_y, test_set_x_orig, test_set_y, classes = load_dataset()

train_set_x_flatten = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T
test_set_x_flatten = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T

train_set_x = train_set_x_flatten / 255.
test_set_x = test_set_x_flatten / 255.

logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=2000, learning_rate=0.01)

loss 0.6931471805599453
loss 1.1201739474561658
loss 2.11271838059857
loss 3.1997290177984428
loss 0.8100261060105536
loss 1.8874109951783222
loss 3.1881676760526965
loss 0.7949783817423247
loss 1.870697417485222
loss 3.1513560070049493
loss 0.7648481658963865
loss 1.802202325089909
loss 3.113051478606344
loss 0.7356117417067404
loss 1.721601153821593
loss 3.0698415680999123
loss 0.7051661626075594
loss 1.6227657065309706
loss 3.0182244725746434
loss 0.6719081806153118
loss 1.4963431351303784
loss 2.949022059586552
loss 0.6323339800435082
loss 1.3139214953075353
loss 2.8283499427181225
loss 0.5775558543072069
loss 0.9667092700909913
loss 2.4168797028465243
loss 0.5594224794650073
loss 0.8473694424360803
loss 1.848507416944733
loss 2.914995244640351
loss 0.6061398574888199
loss 1.236423200116335
loss 2.696790061688035
loss 0.5264808290812679
loss 0.6582804224087035
loss 1.5344517149021661
loss 1.2992095821532161
loss 2.7144409289813685
loss 0.5234222868466314
loss 0.7190704405345179
los